# Day 2 Group Exercise: Data Quality Check

In-class group notebook. Focus on inspection, KPI measurement, validation, and reporting.

**Group members:** Sara, Luca, Mehmet, Jana  
**Dataset:** `week2_customer_transactions_messy.csv`


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

df = pd.read_csv('week2_customer_transactions_messy.csv')
print('Shape:', df.shape)
df.head(10)


##  tasks

1. inspect the dataset
2. identify major data quality issues
3. map issues to quality dimensions
4. calculate at least 3 KPIs
5. define 3 validation rules
6. summarize findings
7. suggest cleaning actions for next week


## Step 1 - Inspect dataset


In [ ]:
df.info()

# Add extra checks here
print('\nmissing values per column ')
print(df.isna().sum())

print('\n unique values for categoricals ')
for col in ['currency', 'payment_method', 'status', 'region']:
    print(f'{col}: {df[col].unique().tolist()}')

print('\n numeric summary ')
df.describe()


### Write your observations here

- **4 columns have missing values:** customer_id (1 missing), amount (1), payment_method (1), last_updated (1). Not huge numbers but customer_id and amount are critical fields.
- **Duplicate row:** transaction_id T0006 appears twice — looks like the same record got inserted twice. This is bad for any aggregation.
- **Date format mess:** transaction_date uses at least 3 different formats (`2026-01-05`, `2026/01/06`, `06-01-2026`). One date (`2026-02-30`) is literally impossible — Feb doesn't have 30 days.
- **Amount issues:** T0003 has a negative amount (-35), T0002 has zero, and T0008 has 1,000,000 which seems way too high — possible data entry error or fraud?
- **Currency:** `EURO` is not a valid ISO code, should be `EUR`.
- **Casing problems:** payment_method has both `card` and `CARD`, region has `DE` and `de`. Same values, different cases — will mess up any groupby.


## Step 2 - Map issues to dimensions


In [ ]:
# built on the template from the notebook
issues = pd.DataFrame([
    ['Missing customer_id (T0004)',           'Completeness', 'High',   'Cannot link transaction to customer — breaks CRM and segmentation'],
    ['Missing amount (T0009)',                'Completeness', 'High',   'Revenue cannot be recorded, financial reports will be wrong'],
    ['Missing payment_method (T0010)',        'Completeness', 'Medium', 'Payment channel analysis and fraud detection incomplete'],
    ['Missing last_updated (T0009)',          'Completeness', 'Low',    'Audit trail broken, cannot track when record was last changed'],
    ['Duplicate transaction_id T0006',        'Uniqueness',   'High',   'Double-counting in revenue reports, violates primary key constraint'],
    ['Inconsistent date formats in tx_date',  'Validity',     'High',   'Parsing fails or gives wrong dates silently'],
    ['Impossible date 2026-02-30 (T0007)',    'Validity',     'High',   'Row cannot be placed on timeline, all date-based analysis breaks'],
    ['Negative amount -35.00 (T0003)',        'Validity',     'High',   'Invalid for a transaction, distorts revenue and fraud metrics'],
    ['Zero amount 0.00 (T0002)',              'Validity',     'Medium', 'Likely test or ghost record, inflates transaction count'],
    ['Extreme outlier 1,000,000 (T0008)',     'Validity',     'Medium', 'Possible data entry error or fraud, needs manual review'],
    ['Invalid currency code EURO (T0005)',    'Validity',     'Medium', 'Not ISO 4217 compliant, FX conversion will fail'],
    ['Mixed case payment_method (CARD/card)', 'Consistency',  'Medium', 'Same value treated as two categories, splits aggregations'],
    ['Mixed case region (DE/de)',             'Consistency',  'Low',    'Regional grouping splits, dashboards show duplicate labels'],
    ['last_updated far after tx_date (T0006)','Integrity',    'Medium', 'Temporal logic violated for a cancelled record'],
], columns=['Issue', 'Dimension', 'Severity', 'Why it matters'])

issues


### Add your mapping here

We found issues across all 5 quality dimensions. **Validity** has the most (6 issues) and also the highest financial risk since bad amounts go directly into revenue numbers. Completeness comes second with 4 issues but two of those (missing customer_id and amount) are high severity.

Consistency issues like casing are annoying but lower risk — they don't corrupt data, just make groupbys messy.


## Step 3 - Calculate at least 3 KPIs


In [ ]:
# KPI 1: overall completeness
total_cells = df.shape[0] * df.shape[1]
completeness = 1 - (df.isna().sum().sum() / total_cells)

# KPI 2: duplication rate on transaction_id
dup_rate = df.duplicated(subset=['transaction_id']).mean()

# KPI 3: amount validity rate (must be numeric, positive, and below 10k plausibility cap)
amount_num = pd.to_numeric(df['amount'], errors='coerce')
valid_amount = amount_num.notna() & (amount_num > 0) & (amount_num <= 10_000)
amount_validity = valid_amount.mean()

# KPI 4: currency standardisation rate
valid_ccy = ['EUR', 'USD', 'GBP']
ccy_rate = df['currency'].str.upper().str.strip().isin(valid_ccy).mean()

# KPI 5: date parseability
tx_dt = pd.to_datetime(df['transaction_date'], errors='coerce', format='mixed')
date_ok = tx_dt.notna().mean()

pd.DataFrame({
    'KPI': ['Completeness rate', 'Duplication rate (transaction_id)',
            'Amount validity rate (>0 and <=10k)', 'Currency standardisation rate', 'Date parseability rate'],
    'Value': [completeness, dup_rate, amount_validity, ccy_rate, date_ok],
    'Value (%)': [f'{x*100:.1f}%' for x in [completeness, dup_rate, amount_validity, ccy_rate, date_ok]]
})


### Add your KPI summary here

- **Completeness: 96.0%** — on the surface looks ok but the 4 missing cells include amount and customer_id which are high severity
- **Duplication rate: 9.1%** — 1 in 11 rows is a duplicate. Even one duplicate on a primary key is unacceptable
- **Amount validity: 72.7%** — worst KPI. Over a quarter of amounts fail basic rules (negative, zero, or extreme outlier)
- **Currency standardisation: 90.9%** — one bad currency code. Would silently break FX pipelines
- **Date parseability: 90.9%** — one impossible date plus 3 different formats across the column

**Interpretation:** The 96% completeness is misleading — it hides that the missing fields are among the most important ones. Amount validity at 72.7% is the most alarming number because amount feeds directly into financial KPIs.


## Step 4 - Define 3 validation rules


In [ ]:
# Rule 1: transaction_id must be non-null AND unique
rule_id_missing = df['transaction_id'].isna() | (df['transaction_id'].astype(str).str.strip() == '')
rule_id_dup = df.duplicated(subset=['transaction_id'], keep=False)
rule1 = rule_id_missing | rule_id_dup

# Rule 2: amount must be positive and below 10,000
amount_num = pd.to_numeric(df['amount'], errors='coerce')
rule2 = amount_num.isna() | (amount_num <= 0) | (amount_num > 10_000)

# Rule 3: currency must be a valid ISO 4217 code
valid_ccy = ['EUR', 'USD', 'GBP']
rule3 = ~df['currency'].str.upper().str.strip().isin(valid_ccy)

rules = {
    'transaction_id non-null and unique': rule1,
    'amount positive and <= 10000':        rule2,
    'currency valid ISO 4217 code':        rule3,
}
pd.DataFrame({k: int(v.sum()) for k, v in rules.items()}, index=['affected_rows']).T


### Which issue should be prioritized first and why?

**The duplicate transaction_id should be fixed first.**

A duplicate primary key breaks the entire table's trustworthiness. Every downstream join, aggregation, or report built on transaction_id is wrong from the start. It also has a clear fix — drop the exact duplicate and keep the first occurrence. The amount issues are more serious in terms of business impact (wrong revenue numbers) but you can't even trust the amount analysis if row identity is broken.

Order of priority:
1. Deduplicate on transaction_id
2. Fix / flag bad amounts (negative, zero, extreme outlier)
3. Standardise date formats and nullify impossible dates


## Optional helper function template (for teams)

Use this template if your team wants cleaner, reusable code.
Try changing parameter values to test different assumptions.


In [ ]:
def flag_timeliness_issues(order_dates, ship_dates, max_days=10):
    """Return a boolean mask for timeliness violations.

    Parameters
    ----------
    order_dates : pd.Series (datetime)
        Parsed order creation date.
    ship_dates : pd.Series (datetime)
        Parsed shipping date.
    max_days : int, default=10
        Maximum acceptable shipping delay in days.

    Returns
    -------
    pd.Series (bool)
        True where timeliness rule is violated.
    """
    valid = order_dates.notna() & ship_dates.notna()
    delay = (ship_dates - order_dates).dt.days
    violation = (~valid) | (delay < 0) | (delay > max_days)
    return violation

# we adapted this to check last_updated vs transaction_date instead
tx_dt   = pd.to_datetime(df['transaction_date'], errors='coerce', format='mixed')
upd_dt  = pd.to_datetime(df['last_updated'],     errors='coerce', format='mixed')
integrity_issues = flag_timeliness_issues(tx_dt, upd_dt, max_days=30)
print('Integrity violations (last_updated > 30d after tx):', int(integrity_issues.sum()))


## Final group summary

### Dataset
Customer payment transactions (11 rows x 9 columns). Likely from an e-commerce or fintech platform. Data quality issues here directly affect revenue reporting, fraud detection, and regulatory compliance.

### Key issues
We found 14 issues across all 5 quality dimensions. The biggest problem areas are:
- **Validity (6 issues):** bad dates, negative/zero/extreme amounts, wrong currency code
- **Completeness (4 issues):** missing customer_id and amount are the critical ones
- **Uniqueness (1 issue):** one duplicated transaction_id — small count but high impact

### KPI highlights
| KPI | Value | Status |
|---|---|---|
| Completeness rate | 96.0% | OK on surface, hides critical gaps |
| Duplication rate | 9.1% | Bad — even 1 duplicate is too many |
| Amount validity | 72.7% | Worst KPI — over 25% of amounts are wrong |
| Currency standard | 90.9% | One bad code would break FX pipelines |
| Date parseability | 90.9% | One impossible date + format inconsistency |

### Cleaning recommendations for next week
1. **Deduplicate** on transaction_id (drop exact duplicate rows)
2. **Standardise all dates** to ISO 8601 (YYYY-MM-DD); nullify 2026-02-30
3. **Handle bad amounts** — move negatives to a refunds table, drop zero amounts, flag the 1M outlier for manual review
4. **Fix currency codes** — map EURO → EUR, uppercase all, validate against ISO 4217
5. **Lowercase payment_method, uppercase region codes** — enforce controlled vocabulary
6. **Impute or flag missing fields** — especially customer_id and amount
7. **Assert referential integrity** after cleaning: last_updated must be >= transaction_date
